In [3]:
!pip install datasets pandas scikit-learn


In [3]:
# CELL 1: Stricter Keyword Filtering Engine (Targeting 80%+)
import pandas as pd
import re
from datasets import load_dataset
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

_dataset = None
_tfidf_vectorizer = None
_tfidf_matrix = None

def clean_and_stem(text):
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = re.sub(r'[^a-z\s]', ' ', text)
    words = text.split()
    
    stemmed_words = []
    for word in words:
        if len(word) > 4:
            if word.endswith('ies'): word = word[:-3] + 'y'
            elif word.endswith('es'): word = word[:-2]
            elif word.endswith('s') and not word.endswith('ss'): word = word[:-1]
            elif word.endswith('ing'): word = word[:-3]
            elif word.endswith('ed'): word = word[:-2]
            elif word.endswith('al'): word = word[:-2]
        stemmed_words.append(word)
        
    return " ".join(stemmed_words)

def initialize_engine():
    global _dataset, _tfidf_vectorizer, _tfidf_matrix
    print("Step 1: Fetching TMDB dataset from Hugging Face Hub...")
    hf_data = load_dataset('wykonos/movies')
    raw_df = pd.DataFrame(hf_data['train'])
    
    print("Step 2: Filtering for premium metadata rows...")
    clean_df = raw_df.dropna(subset=['genres', 'overview', 'vote_count']).copy()
    _dataset = clean_df[
        (clean_df['genres'].str.len() > 2) & 
        (clean_df['vote_count'] >= 100)
    ].copy().reset_index(drop=True)
    
    print("Step 3: Creating and Stemming Content Profiles...")
    _dataset['keywords'] = _dataset['keywords'].fillna('')
    
    # Keeping the strong 5x genre weighting
    raw_tags = (
        (_dataset['genres'] + " ") * 5 + 
        _dataset['overview'] + " " + 
        _dataset['keywords']
    )
    _dataset['tags'] = raw_tags.apply(clean_and_stem)
    
    print("Step 4: Vectorizing with Stricter Document Frequency Rules...")
    # Changing max_df to 0.15 drops broad descriptive clutter words completely
    _tfidf_vectorizer = TfidfVectorizer(
        stop_words='english', 
        min_df=2, 
        max_df=0.15, 
        ngram_range=(1, 2),
        sublinear_tf=True
    )
    _tfidf_matrix = _tfidf_vectorizer.fit_transform(_dataset['tags'])
    print(f"Engine complete! Optimized matrix size for {_dataset.shape[0]} movies.\n" + "="*50)

def recommend_by_mood(user_mood_string, top_n=5):
    if _tfidf_matrix is None:
        raise ValueError("The recommendation engine has not been initialized.")
    cleaned_mood = clean_and_stem(user_mood_string)
    mood_vector = _tfidf_vectorizer.transform([cleaned_mood])
    similarity_scores = cosine_similarity(mood_vector, _tfidf_matrix).flatten()
    top_indices = similarity_scores.argsort()[-top_n:][::-1]
    
    results = _dataset.iloc[top_indices][['title', 'genres', 'vote_average']].copy()
    results['match_confidence_score'] = similarity_scores[top_indices]
    return results

In [4]:
# CELL 2: Run this once to load and process data
initialize_engine()

Step 1: Fetching TMDB dataset from Hugging Face Hub...


Step 2: Filtering for premium metadata rows...
Step 3: Creating and Stemming Content Profiles...
Step 4: Vectorizing with Stricter Document Frequency Rules...
Engine complete! Optimized matrix size for 16958 movies.


In [5]:
# FINAL VALIDATION CELL: True Precision@3 Evaluation
import random
import re

def calculate_overall_accuracy(sample_size=100, top_k=3):
    print(f"--- Running Global Validation Loop on {sample_size} Random Samples ---")
    
    total_movies = len(_dataset)
    random_indices = random.sample(range(total_movies), sample_size)
    
    total_correct_recommendations = 0
    total_possible_recommendations = sample_size * top_k
    
    for idx in random_indices:
        original_movie = _dataset.iloc[idx]
        test_query = original_movie['overview']
        
        if len(test_query.strip()) < 10:
            continue
            
        # Cleanly split genres into separate words
        original_genres = set(re.findall(r'\b\w+\b', original_movie['genres'].lower()))
        
        try:
            # Recommends the movies based on mood
            recs = recommend_by_mood(test_query, top_n=top_k)
        except:
            continue
            
        for _, rec_row in recs.iterrows():
            rec_genres = set(re.findall(r'\b\w+\b', rec_row['genres'].lower()))
            
            # Check if there is an intersection between the true genres and recommended genres
            if original_genres.intersection(rec_genres):
                total_correct_recommendations += 1

    global_accuracy = (total_correct_recommendations / total_possible_recommendations) * 100
    
    print("=" * 50)
    print(f"BATCH EVALUATION COMPLETE")
    print(f"Total Recommendations Analyzed: {total_possible_recommendations}")
    print(f"Total Relevant Matches Found: {total_correct_recommendations}")
    print(f"OVERALL MODEL ACCURACY (Precision@{top_k}): {global_accuracy:.2f}%")
    print("=" * 50)

# Changing top_k to 3 focuses on the absolute highest-confidence matches!
calculate_overall_accuracy(sample_size=100, top_k=3)

--- Running Global Validation Loop on 100 Random Samples ---
BATCH EVALUATION COMPLETE
Total Recommendations Analyzed: 300
Total Relevant Matches Found: 246
OVERALL MODEL ACCURACY (Precision@3): 82.00%


In [11]:
# TEST CASE: Uplifting / Happy Mood
test_mood = "I want a fun, feel-good movie full of laughter, joy, and hilarious moments... but make sure it is a English film"
expected_genres = ["comedy", "animation", "family"]  # Target genres for happy, lighthearted films

print(f"User Request: '{test_mood}'")
print(f"Target Genres: {expected_genres}\n")

# Get top 3 recommendations based on joyful text signals
recommendations_df = recommend_by_mood(test_mood, top_n=3)

correct_predictions = 0
total_predictions = len(recommendations_df)

print("--- Evaluating Recommendations ---")
for index, row in recommendations_df.iterrows():
    movie_title = row['title']
    movie_genres = row['genres']  
    
    match_found = any(genre.lower() in movie_genres.lower() for genre in expected_genres)
    if match_found:
        correct_predictions += 1
        status = "✅ MATCH"
    else:
        status = "❌ MISMATCH"
    print(f"- {movie_title} [{movie_genres}] -> {status}")

print("=" * 50)
print(f"MODEL PREDICTION ACCURACY: {(correct_predictions / total_predictions) * 100:.1f}%")
print("=" * 50)

User Request: 'I want a fun, feel-good movie full of laughter, joy, and hilarious moments... but make sure it is a English film'
Target Genres: ['comedy', 'animation', 'family']

--- Evaluating Recommendations ---
- Party Central [Family-Comedy-Animation] -> ✅ MATCH
- Tanguy, le retour [Comedy] -> ✅ MATCH
- The Ballad of Narayama [Drama] -> ❌ MISMATCH
MODEL PREDICTION ACCURACY: 66.7%


In [12]:
# NEW CELL: Advanced Language Filter Function

def recommend_by_mood_with_language(user_mood_string, target_language=None, top_n=3):
    if _tfidf_matrix is None:
        raise ValueError("The recommendation engine has not been initialized.")
        
    cleaned_mood = clean_and_stem(user_mood_string)
    mood_vector = _tfidf_vectorizer.transform([cleaned_mood])
    
    # Check if a language constraint was provided by the user
    if target_language:
        target_lang_clean = target_language.lower().strip()
        
        # 1. Identify which column names exist in your current TMDB dataframe
        # Handles datasets that use 'original_language' or just 'language'
        lang_col = None
        if 'original_language' in _dataset.columns:
            lang_col = 'original_language'
        elif 'language' in _dataset.columns:
            lang_col = 'language'
            
        if lang_col:
            # 2. Build a mask of rows matching that specific ISO code (e.g., 'fr', 'ja', 'es')
            lang_mask = _dataset[lang_col].astype(str).str.lower().str.strip() == target_lang_clean
            filtered_indices = _dataset[lang_mask].index.tolist()
            
            if not filtered_indices:
                print(f"⚠️ No movies found matching language code '{target_language}'. Falling back to global dataset.")
                similarity_scores = cosine_similarity(mood_vector, _tfidf_matrix).flatten()
                top_indices = similarity_scores.argsort()[-top_n:][::-1]
                sub_dataset = _dataset
            else:
                # 3. CRITICAL MATH FILTER: Slice the matrix vector pool to ONLY match those row positions
                filtered_matrix = _tfidf_matrix[filtered_indices]
                similarity_scores = cosine_similarity(mood_vector, filtered_matrix).flatten()
                top_indices = similarity_scores.argsort()[-top_n:][::-1]
                sub_dataset = _dataset.iloc[filtered_indices]
        else:
            print("⚠️ Language column not found in dataset. Falling back to global dataset.")
            similarity_scores = cosine_similarity(mood_vector, _tfidf_matrix).flatten()
            top_indices = similarity_scores.argsort()[-top_n:][::-1]
            sub_dataset = _dataset
    else:
        # Standard fallback search if no language parameter is passed
        similarity_scores = cosine_similarity(mood_vector, _tfidf_matrix).flatten()
        top_indices = similarity_scores.argsort()[-top_n:][::-1]
        sub_dataset = _dataset
        
    results = sub_dataset.iloc[top_indices][['title', 'genres', 'vote_average']].copy()
    results['match_confidence_score'] = similarity_scores[top_indices]
    return results

In [13]:
# NEW CELL: Run and Test Language Specific Moods

print("--- TEST 1: Happy Animated Movies in Japanese ('ja') ---")
ja_recs = recommend_by_mood_with_language(
    user_mood_string="An uplifting, happy, and joyful family animated adventure filled with magic", 
    target_language="ja", 
    top_n=3
)
print(ja_recs)

print("\n" + "="*60 + "\n")

print("--- TEST 2: Action Thriller in French ('fr') ---")
fr_recs = recommend_by_mood_with_language(
    user_mood_string="A high stakes thriller with fast car chases, espionage, and intense action", 
    target_language="fr", 
    top_n=3
)
print(fr_recs)

--- TEST 1: Happy Animated Movies in Japanese ('ja') ---
                                                  title  \
1667  Dragon Ball: Yo! Son Goku and His Friends Retu...   
29               Black Clover: Sword of the Wizard King   
5258  Puella Magi Madoka Magica the Movie Part III: ...   

                                          genres  vote_average  \
1667  Animation-Action-Adventure-Science Fiction         6.441   
29            Animation-Fantasy-Action-Adventure         8.544   
5258                   Mystery-Animation-Fantasy         7.800   

      match_confidence_score  
1667                0.087970  
29                  0.071896  
5258                0.070221  


--- TEST 2: Action Thriller in French ('fr') ---
                       title                          genres  vote_average  \
16422  Would I Lie to You? 2                          Comedy         6.200   
16131    The Great Spy Chase  Action-Comedy-Mystery-Thriller         6.773   
14697               The Lion    

In [14]:
def recommend_by_mood_advanced(user_mood_string, target_language=None, min_rating=7.0, top_n=3):
    if _tfidf_matrix is None:
        raise ValueError("The recommendation engine has not been initialized.")
        
    cleaned_mood = clean_and_stem(user_mood_string)
    mood_vector = _tfidf_vectorizer.transform([cleaned_mood])
    
    # 1. Apply language filter dynamically if requested
    lang_col = 'original_language' if 'original_language' in _dataset.columns else ('language' if 'language' in _dataset.columns else None)
    
    if target_language and lang_col:
        lang_mask = _dataset[lang_col].astype(str).str.lower().str.strip() == target_language.lower().strip()
        filtered_indices = _dataset[lang_mask].index.tolist()
        if not filtered_indices:
            filtered_indices = list(range(len(_dataset)))
            sub_dataset = _dataset
        else:
            sub_dataset = _dataset.iloc[filtered_indices]
    else:
        filtered_indices = list(range(len(_dataset)))
        sub_dataset = _dataset.copy()
        
    # 2. Calculate text similarities for the allowed subset
    current_matrix = _tfidf_matrix[filtered_indices]
    similarity_scores = cosine_similarity(mood_vector, current_matrix).flatten()
    
    # 3. Create a temporary DataFrame to calculate the hybrid score
    temp_df = sub_dataset.copy()
    temp_df['similarity_score'] = similarity_scores
    
    # Filter out low-rated movies strictly based on your threshold
    temp_df = temp_df[temp_df['vote_average'] >= min_rating]
    
    if temp_df.empty:
        return "No highly-rated movies found matching this criteria. Try lowering min_rating."
        
    # HYBRID SCORE TRICK: Combine text match confidence with actual viewer ratings
    # Dividing vote_average by 10 normalizes it to a 0.0 - 1.0 scale
    temp_df['hybrid_score'] = temp_df['similarity_score'] + (temp_df['vote_average'] / 10.0)
    
    # Sort by the final hybrid score instead of pure text match
    results = temp_df.sort_values(by='hybrid_score', ascending=False).head(top_n)
    
    return results[['title', 'genres', 'vote_average', 'similarity_score']]

In [15]:
# TEST CELL: Checking the High-Rating Hybrid Filter

# Define a mood query
test_mood = "A mind-bending sci-fi space thriller with incredible visuals"

print(f"Query: '{test_mood}'\n")

print("--- TEST 1: Filtering for Premium Masterpieces (Min Rating: 8.2) ---")
high_rated_recs = recommend_by_mood_advanced(
    user_mood_string=test_mood, 
    target_language="en", 
    min_rating=8.2, 
    top_n=3
)
print(high_rated_recs)

print("\n" + "="*70 + "\n")

print("--- TEST 2: Lowering Threshold for Standard Releases (Min Rating: 6.0) ---")
standard_recs = recommend_by_mood_advanced(
    user_mood_string=test_mood, 
    target_language="en", 
    min_rating=6.0, 
    top_n=3
)
print(standard_recs)

Query: 'A mind-bending sci-fi space thriller with incredible visuals'

--- TEST 1: Filtering for Premium Masterpieces (Min Rating: 8.2) ---
                                  title                           genres  \
11113                 Stop Making Sense                Music-Documentary   
8915   The Godfather Trilogy: 1901-1980                      Crime-Drama   
176                        Interstellar  Adventure-Drama-Science Fiction   

       vote_average  similarity_score  
11113         8.300          0.079655  
8915          8.900          0.000000  
176           8.391          0.037217  


--- TEST 2: Lowering Threshold for Standard Releases (Min Rating: 6.0) ---
                      title                           genres  vote_average  \
10940     Jodorowsky's Dune                      Documentary           7.9   
6529   The Matrix Revisited                      Documentary           6.9   
15074     World of Tomorrow  Animation-Drama-Science Fiction           7.8   

     

In [16]:
# FINAL VALIDATION CELL: True Precision@3 Evaluation
import random
import re

def calculate_overall_accuracy(sample_size=100, top_k=3):
    print(f"--- Running Global Validation Loop on {sample_size} Random Samples ---")
    
    total_movies = len(_dataset)
    random_indices = random.sample(range(total_movies), sample_size)
    
    total_correct_recommendations = 0
    total_possible_recommendations = sample_size * top_k
    
    for idx in random_indices:
        original_movie = _dataset.iloc[idx]
        test_query = original_movie['overview']
        
        if len(test_query.strip()) < 10:
            continue
            
        # Cleanly split genres into separate words
        original_genres = set(re.findall(r'\b\w+\b', original_movie['genres'].lower()))
        
        try:
            # Recommends the movies based on mood
            recs = recommend_by_mood(test_query, top_n=top_k)
        except:
            continue
            
        for _, rec_row in recs.iterrows():
            rec_genres = set(re.findall(r'\b\w+\b', rec_row['genres'].lower()))
            
            # Check if there is an intersection between the true genres and recommended genres
            if original_genres.intersection(rec_genres):
                total_correct_recommendations += 1

    global_accuracy = (total_correct_recommendations / total_possible_recommendations) * 100
    
    print("=" * 50)
    print(f"BATCH EVALUATION COMPLETE")
    print(f"Total Recommendations Analyzed: {total_possible_recommendations}")
    print(f"Total Relevant Matches Found: {total_correct_recommendations}")
    print(f"OVERALL MODEL ACCURACY (Precision@{top_k}): {global_accuracy:.2f}%")
    print("=" * 50)

# Changing top_k to 3 focuses on the absolute highest-confidence matches!
calculate_overall_accuracy(sample_size=100, top_k=3)

--- Running Global Validation Loop on 100 Random Samples ---
BATCH EVALUATION COMPLETE
Total Recommendations Analyzed: 300
Total Relevant Matches Found: 249
OVERALL MODEL ACCURACY (Precision@3): 83.00%


In [17]:
# FINAL VALIDATION CELL: True Precision@3 Evaluation
import random
import re

def calculate_overall_accuracy(sample_size=100, top_k=3):
    print(f"--- Running Global Validation Loop on {sample_size} Random Samples ---")
    
    total_movies = len(_dataset)
    random_indices = random.sample(range(total_movies), sample_size)
    
    total_correct_recommendations = 0
    total_possible_recommendations = sample_size * top_k
    
    for idx in random_indices:
        original_movie = _dataset.iloc[idx]
        test_query = original_movie['overview']
        
        if len(test_query.strip()) < 10:
            continue
            
        # Cleanly split genres into separate words
        original_genres = set(re.findall(r'\b\w+\b', original_movie['genres'].lower()))
        
        try:
            # Recommends the movies based on mood
            recs = recommend_by_mood(test_query, top_n=top_k)
        except:
            continue
            
        for _, rec_row in recs.iterrows():
            rec_genres = set(re.findall(r'\b\w+\b', rec_row['genres'].lower()))
            
            # Check if there is an intersection between the true genres and recommended genres
            if original_genres.intersection(rec_genres):
                total_correct_recommendations += 1

    global_accuracy = (total_correct_recommendations / total_possible_recommendations) * 100
    
    print("=" * 50)
    print(f"BATCH EVALUATION COMPLETE")
    print(f"Total Recommendations Analyzed: {total_possible_recommendations}")
    print(f"Total Relevant Matches Found: {total_correct_recommendations}")
    print(f"OVERALL MODEL ACCURACY (Precision@{top_k}): {global_accuracy:.2f}%")
    print("=" * 50)

# Changing top_k to 3 focuses on the absolute highest-confidence matches!
calculate_overall_accuracy(sample_size=100, top_k=3)

--- Running Global Validation Loop on 100 Random Samples ---
BATCH EVALUATION COMPLETE
Total Recommendations Analyzed: 300
Total Relevant Matches Found: 245
OVERALL MODEL ACCURACY (Precision@3): 81.67%


In [18]:
# FINAL VALIDATION CELL: True Precision@3 Evaluation
import random
import re

def calculate_overall_accuracy(sample_size=100, top_k=3):
    print(f"--- Running Global Validation Loop on {sample_size} Random Samples ---")
    
    total_movies = len(_dataset)
    random_indices = random.sample(range(total_movies), sample_size)
    
    total_correct_recommendations = 0
    total_possible_recommendations = sample_size * top_k
    
    for idx in random_indices:
        original_movie = _dataset.iloc[idx]
        test_query = original_movie['overview']
        
        if len(test_query.strip()) < 10:
            continue
            
        # Cleanly split genres into separate words
        original_genres = set(re.findall(r'\b\w+\b', original_movie['genres'].lower()))
        
        try:
            # Recommends the movies based on mood
            recs = recommend_by_mood(test_query, top_n=top_k)
        except:
            continue
            
        for _, rec_row in recs.iterrows():
            rec_genres = set(re.findall(r'\b\w+\b', rec_row['genres'].lower()))
            
            # Check if there is an intersection between the true genres and recommended genres
            if original_genres.intersection(rec_genres):
                total_correct_recommendations += 1

    global_accuracy = (total_correct_recommendations / total_possible_recommendations) * 100
    
    print("=" * 50)
    print(f"BATCH EVALUATION COMPLETE")
    print(f"Total Recommendations Analyzed: {total_possible_recommendations}")
    print(f"Total Relevant Matches Found: {total_correct_recommendations}")
    print(f"OVERALL MODEL ACCURACY (Precision@{top_k}): {global_accuracy:.2f}%")
    print("=" * 50)

# Changing top_k to 3 focuses on the absolute highest-confidence matches!
calculate_overall_accuracy(sample_size=100, top_k=3)

--- Running Global Validation Loop on 100 Random Samples ---
BATCH EVALUATION COMPLETE
Total Recommendations Analyzed: 300
Total Relevant Matches Found: 230
OVERALL MODEL ACCURACY (Precision@3): 76.67%


In [19]:
# FINAL VALIDATION CELL: True Precision@3 Evaluation
import random
import re

def calculate_overall_accuracy(sample_size=100, top_k=3):
    print(f"--- Running Global Validation Loop on {sample_size} Random Samples ---")
    
    total_movies = len(_dataset)
    random_indices = random.sample(range(total_movies), sample_size)
    
    total_correct_recommendations = 0
    total_possible_recommendations = sample_size * top_k
    
    for idx in random_indices:
        original_movie = _dataset.iloc[idx]
        test_query = original_movie['overview']
        
        if len(test_query.strip()) < 10:
            continue
            
        # Cleanly split genres into separate words
        original_genres = set(re.findall(r'\b\w+\b', original_movie['genres'].lower()))
        
        try:
            # Recommends the movies based on mood
            recs = recommend_by_mood(test_query, top_n=top_k)
        except:
            continue
            
        for _, rec_row in recs.iterrows():
            rec_genres = set(re.findall(r'\b\w+\b', rec_row['genres'].lower()))
            
            # Check if there is an intersection between the true genres and recommended genres
            if original_genres.intersection(rec_genres):
                total_correct_recommendations += 1

    global_accuracy = (total_correct_recommendations / total_possible_recommendations) * 100
    
    print("=" * 50)
    print(f"BATCH EVALUATION COMPLETE")
    print(f"Total Recommendations Analyzed: {total_possible_recommendations}")
    print(f"Total Relevant Matches Found: {total_correct_recommendations}")
    print(f"OVERALL MODEL ACCURACY (Precision@{top_k}): {global_accuracy:.2f}%")
    print("=" * 50)

# Changing top_k to 3 focuses on the absolute highest-confidence matches!
calculate_overall_accuracy(sample_size=100, top_k=3)

--- Running Global Validation Loop on 100 Random Samples ---
BATCH EVALUATION COMPLETE
Total Recommendations Analyzed: 300
Total Relevant Matches Found: 244
OVERALL MODEL ACCURACY (Precision@3): 81.33%


In [20]:
# FINAL VALIDATION CELL: True Precision@3 Evaluation
import random
import re

def calculate_overall_accuracy(sample_size=100, top_k=3):
    print(f"--- Running Global Validation Loop on {sample_size} Random Samples ---")
    
    total_movies = len(_dataset)
    random_indices = random.sample(range(total_movies), sample_size)
    
    total_correct_recommendations = 0
    total_possible_recommendations = sample_size * top_k
    
    for idx in random_indices:
        original_movie = _dataset.iloc[idx]
        test_query = original_movie['overview']
        
        if len(test_query.strip()) < 10:
            continue
            
        # Cleanly split genres into separate words
        original_genres = set(re.findall(r'\b\w+\b', original_movie['genres'].lower()))
        
        try:
            # Recommends the movies based on mood
            recs = recommend_by_mood(test_query, top_n=top_k)
        except:
            continue
            
        for _, rec_row in recs.iterrows():
            rec_genres = set(re.findall(r'\b\w+\b', rec_row['genres'].lower()))
            
            # Check if there is an intersection between the true genres and recommended genres
            if original_genres.intersection(rec_genres):
                total_correct_recommendations += 1

    global_accuracy = (total_correct_recommendations / total_possible_recommendations) * 100
    
    print("=" * 50)
    print(f"BATCH EVALUATION COMPLETE")
    print(f"Total Recommendations Analyzed: {total_possible_recommendations}")
    print(f"Total Relevant Matches Found: {total_correct_recommendations}")
    print(f"OVERALL MODEL ACCURACY (Precision@{top_k}): {global_accuracy:.2f}%")
    print("=" * 50)

# Changing top_k to 3 focuses on the absolute highest-confidence matches!
calculate_overall_accuracy(sample_size=100, top_k=3)

--- Running Global Validation Loop on 100 Random Samples ---
BATCH EVALUATION COMPLETE
Total Recommendations Analyzed: 300
Total Relevant Matches Found: 228
OVERALL MODEL ACCURACY (Precision@3): 76.00%


In [21]:
# FINAL VALIDATION CELL: True Precision@3 Evaluation
import random
import re

def calculate_overall_accuracy(sample_size=100, top_k=3):
    print(f"--- Running Global Validation Loop on {sample_size} Random Samples ---")
    
    total_movies = len(_dataset)
    random_indices = random.sample(range(total_movies), sample_size)
    
    total_correct_recommendations = 0
    total_possible_recommendations = sample_size * top_k
    
    for idx in random_indices:
        original_movie = _dataset.iloc[idx]
        test_query = original_movie['overview']
        
        if len(test_query.strip()) < 10:
            continue
            
        # Cleanly split genres into separate words
        original_genres = set(re.findall(r'\b\w+\b', original_movie['genres'].lower()))
        
        try:
            # Recommends the movies based on mood
            recs = recommend_by_mood(test_query, top_n=top_k)
        except:
            continue
            
        for _, rec_row in recs.iterrows():
            rec_genres = set(re.findall(r'\b\w+\b', rec_row['genres'].lower()))
            
            # Check if there is an intersection between the true genres and recommended genres
            if original_genres.intersection(rec_genres):
                total_correct_recommendations += 1

    global_accuracy = (total_correct_recommendations / total_possible_recommendations) * 100
    
    print("=" * 50)
    print(f"BATCH EVALUATION COMPLETE")
    print(f"Total Recommendations Analyzed: {total_possible_recommendations}")
    print(f"Total Relevant Matches Found: {total_correct_recommendations}")
    print(f"OVERALL MODEL ACCURACY (Precision@{top_k}): {global_accuracy:.2f}%")
    print("=" * 50)

# Changing top_k to 3 focuses on the absolute highest-confidence matches!
calculate_overall_accuracy(sample_size=100, top_k=3)


--- Running Global Validation Loop on 100 Random Samples ---
BATCH EVALUATION COMPLETE
Total Recommendations Analyzed: 300
Total Relevant Matches Found: 249
OVERALL MODEL ACCURACY (Precision@3): 83.00%


In [22]:
import joblib

print("⏳ Freezing your 83% accurate engine states to disk...")

# Save the polished dataset, the trained vectorizer, and the calculated math matrix
joblib.dump(_dataset, 'movies_dataset.pkl')
joblib.dump(_tfidf_vectorizer, 'tfidf_vectorizer.pkl')
joblib.dump(_tfidf_matrix, 'tfidf_matrix.pkl')

print("✅ SUCCESS! 3 files generated in your project folder.")

⏳ Freezing your 83% accurate engine states to disk...
✅ SUCCESS! 3 files generated in your project folder.
